In [ ]:
import pandas as pd

from analyses.data_readers.recording_metadata_reader import RecordingMetadataReader

In [ ]:
cells = pd.read_excel('all_anova_passed_cells_old.xlsx')
cells

In [ ]:
from analyses.data_readers.recording_metadata_reader import RecordingMetadataReader
reader = RecordingMetadataReader()
prelim = reader.get_metadata_for_preliminary_analysis()
prelim

In [ ]:
prelim['Location'] = prelim['Location'].replace({'Amygdala': 'AMG'})
prelim['Location'] = prelim['Location'].fillna('Unknown')
cells = cells.merge(
    prelim[['Date', 'Round No.', 'Location']],
    on=['Date', 'Round No.'],
    how='left'
)

# 3. 이제 NeuronID 생성
cells['NeuronID'] = (
    cells['Location'].astype(str) + "_" +
    cells['Date'].astype(str) + "_" +
    cells['Round No.'].astype(str) + "_" +
    cells['Cell'].astype(str))

In [ ]:
cells['Time Window'] = cells['Time Window'].apply(
    lambda s: tuple(int(float(num)) for num in s.strip('()').split(',')))

In [ ]:
# 튜플을 각각 분리
cells[['WindowStart_ms', 'WindowEnd_ms']] = cells['Time Window'].apply(pd.Series)


In [ ]:
cells.drop(cells.columns[[0, 1]], axis=1, inplace=True)

In [ ]:
cells

In [ ]:
from analyses.linear_regression.behavior_vector_linear_regression import \
    run_directional_vector_linear_regression_window_level
from analyses.spike_rate import compute_mean_spike_rate_for_windows
import os
from analyses.enums.monkey_names import get_monkeys_by_default_order

monkey_group = "Zombies"
subject_monkey_index = 6
monkey_list = get_monkeys_by_default_order(monkey_group)
base_dir = '/social_data/zombies_social_data/'
behavior_files = {
    "AffiliationTo": "zombies_feature_df_affiliation.xlsx",
    "AffiliationFrom": "zombies_feature_df_affiliation.xlsx",
    "SubmissionTo": "zombies_feature_df_submission.xlsx",
    "SubmissionFrom": "zombies_feature_df_submission.xlsx",
    "AgonismTo": "zombies_feature_df_agonism.xlsx",
    "AgonismFrom": "zombies_feature_df_agonism.xlsx",
}

behavior_matrices = {
    name: pd.read_excel(os.path.join(base_dir, fname)).iloc[:, 1:].to_numpy().T if 'From' in name
    else pd.read_excel(os.path.join(base_dir, fname)).iloc[:, 1:].to_numpy()
    for name, fname in behavior_files.items()
}



In [ ]:
cells

In [ ]:
# Directional Linear Regression (OLS) on Significant Windows
sig_windows = cells
mean_spike_rate_windows = compute_mean_spike_rate_for_windows(sig_windows)
all_window_results = []
for name, mat in behavior_matrices.items():
    results_df = run_directional_vector_linear_regression_window_level(
        mean_spike_rate_windows, mat, name, monkey_group, monkey_list, subject_monkey_index, use_spikerate=True
    )
    all_window_results.append(results_df)

In [ ]:
all_window_results_df = pd.concat(all_window_results, ignore_index=True)

In [ ]:
all_window_results_df

In [ ]:

all_window_results_sorted = all_window_results_df.sort_values(by=['p_value','R-squared'], ascending=False)
filtered_df = all_window_results_df[
    (all_window_results_df['p_value'] < 0.05) &
    (all_window_results_df['R-squared'] > 0.5)
    ]
mean_spike_rate = compute_mean_spike_rate_for_windows(filtered_df)